# 8.2 Institutional Pattern Discovery II: Unsupervised Learning with Survey Data

## Introduction



In the previous sections, we used **supervised** models to predict known outcomes like retention or stress. However, some of the most valuable institutional insights come from **unsupervised learning**, where we don't have a specific "label" to predict.

Instead, we ask the data to reveal the natural, hidden groupings within our student population. This allows us to move beyond broad demographic categories and discover a small number of student **clusters** based on their actual behaviors and voices.

<br>

#### **Guiding Questions**
1. Can we enhance our discovery of natural groupings of students by combining structured academic records with unstructured survey text?
2. How do we interpret and translate these mathematical "clusters" into actionable institutional strategies?

<br>

#### **Learning Objectives**
By the end of this notebook, you will be able to:
* **Apply K-Means Clustering** to find latent student segments.
* **Determine the Optimal K** using the Elbow Method.
* **Profile and Interpret** clusters to communicate high-impact findings to campus stakeholders.


## 1. Why Unsupervised Learning in IR?


Supervised learning tells us *who* is likely to leave or succeed - simple categories. Unsupervised learning unearths deeper combinations of attributes telling us **who our students actually are**.

By using **K-Means Clustering**, we can surface distinct segments that might otherwise remain hidden in large datasets. Some hypothetical examples could be:
* **The High-Engagement Thrivers:** Students with high GPAs and positive survey sentiment who may be perfect candidates for peer-mentorship roles.
* **The "Silent" At-Risk Group:** Students with moderate grades but low survey engagement: a subtle early-warning signal that administrative data often misses.
* **The Transition-Strugglers:** First-gen or transfer students whose open-ended comments reflect high academic anxiety despite stable mid-term grades.

> **Institutional Perspective:** Clustering turns "Big Data" into "Human Personas." Instead of looking at 5,000 individual rows, we can look at 3 or 4 meaningful student profiles, allowing for much more personalized and effective resource allocation.

## 2. Setup and Data Preparation Import

Let's import the usual libraries as well as our machine learning ready data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.options.display.max_columns = None
np.random.seed(15)
random.seed(15)


You may recall that free-response text vectorized with TF-IDF produces hundreds of columns, far too many to cluster directly alongside our ~19 structured features. To prevent the text data from mathematically "overwhelming" our academic metrics, we used **PCA (Principal Component Analysis)**.

PCA compresses the high-dimensional word matrix into a small number of "Principal Components" that capture the core variance (the "essence") of the student voice. That led to the following


In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
df_ml_train = pd.read_csv(f'{filepath}ML_SURVEY_MASTER_TRAIN_SEM2.csv')

df_ml_train

> **Instructor Perspective:** Think of PCA as an "Equalizer." Without it, the 250+ word columns would act like 250+ separate votes in the clustering algorithm, while your GPA only gets one vote. PCA condenses those 250+ votes into roughly 30, giving the "Student Voice" and "Student Records" a more balanced influence on the final personas.


## 3. Identifying the Feature Matrix

Let's create variables that will hold our label and features.

In [ ]:
X_train = df_ml_train.drop(columns = ['SEM_2_STATUS'])

To be able to identify characteristics of students in each cluster, it is helpful to have a version of the DataFrame before the demographic and academic variables are preprocessed.

In [ ]:
df_training_assignment = pd.read_csv('/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/data/training_assignment.csv')
df_training1 = df_training_assignment.drop(columns = ['SEM_2_STATUS'])
X_raw = pd.concat([df_training1,X_train.iloc[:,19:]],axis=1)
X_raw


## 6. Choose Number of Clusters: Elbow Method


As you may recall, the K-Means algorithm requires us to specify $K$ (the number of clusters) upfront. Because there is no "correct" label in unsupervised learning, we use the **Elbow Method** to find a mathematically sound value for $K$.

We plot **Inertia** (the sum of squared distances from each student point to their assigned cluster center). As $K$ increases, inertia naturally drops (why?). We are looking for the "Elbow"—the specific point where adding more clusters no longer yields a significant decrease in inertia.



> **Instructor Perspective:** In Institutional Research, choosing $K$ is a balance between **Granularity** and **Actionability**. A $K$ of 20 might be mathematically precise, but a Provost cannot design 20 different student success initiatives. We typically look for $K$ values between 3 and 6 to ensure the resulting personas are distinct enough to drive real campus policy.


We iterate through a range of potential cluster counts ($K=1$ to $10$), calculating the inertia at each step to visualize the trade-off between model complexity and group cohesion.

In [ ]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train)
    inertia.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertia,
              markers=True,
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'},
              title='Elbow Method — Choosing K for K-Means')
fig.show()
print("Look for the point where the curve flattens — that is your elbow.")


## 7. Fit K-Means & Assign Cluster Labels





Visually, the "flatening" occurs in the $K=3-5$ range. We'll be conservative and choose $K=3$. Now that we have identified the "Elbow" at $K=3$, we fit our final model. This process assigns every student in our Fall 2019 cohort to one of three distinct groups based on the mathematical similarities in their academic records and survey voices.

> **Technical Note:** We use `n_init=10` to ensure the model runs multiple times with different starting points, selecting the best result to avoid "local minima" (sub-optimal groupings).


In [ ]:
OPTIMAL_K = 3  # ← update this based on the elbow plot above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans.fit(X_train)

X_train = X_train.copy()
X_raw['Cluster'] = kmeans.labels_

print("Cluster sizes:")
print(X_raw['Cluster'].value_counts().sort_index())


## 8. Visualize Clusters with PCA



Our master matrix has **52 dimensions** (19 structured features + 33 text components). Because the human eye cannot see in 52D, we use PCA once more to project the data down to **two coordinates (PC1 and PC2)** for visualization.

**Important Distinction:** * The **Clustering** was done using all 52 features to ensure maximum accuracy.
* The **Visualization** uses only 2 features so we can "see" the separation between our student groups.



> **Key note:** Look for the "overlap" in the scatter plot. In social science data, clusters are rarely perfectly isolated circles; students are complex and often sit on the borders between personas. Our goal is to find the **dominant trend** in each group.



In [ ]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(X_train)

df_plot = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Cluster': X_raw['Cluster'].astype(str),
    'GPA': X_raw['HS_GPA'].round(2),
    'First_Gen': X_raw['FIRST_GEN_STATUS'],
    'Gender': X_raw['GENDER']
}, index=X_raw.index)

fig = px.scatter(
    df_plot, x='PC1', y='PC2',
    color='Cluster',
    hover_data=['GPA', 'First_Gen', 'Gender'],
    title=f'K-Means Clusters (K={OPTIMAL_K}) — Visualized with 2D PCA',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'}
)
fig.update_traces(marker=dict(size=6, opacity=0.75))
fig.show()


Here we can see three distinct groupings geometrically. However there is not a natural seperation between the clusters. This may imply students on the borders are virtually indistinguishible in terms of cluster membership.


## 9. Profile and Interpret Clusters


The scatter plot shows *where* clusters sit in 2D space, but to take action, we must understand *who* is in each group. We "profile" these clusters by calculating the average values for our key academic and demographic features.

In Institutional Research, this step is known as **Persona Development**. We are looking for the "Typical Student" within each cluster so we can tailor support services to their specific needs.



> **Instructor Perspective:** Don't just look at the numbers—look for the **Story**. If Cluster 2 has a high DFW rate and mentions "work-life balance" in their comments, they aren't just a "low-performing group"; they are a "High-Obligation" group that may need more flexible course scheduling.


We calculate the mean for each academic metric by cluster to identify the performance profile of each student group.

In [ ]:
# Numeric profile
profile_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
                'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
profile_cols = [c for c in profile_cols if c in X_raw.columns]

cluster_profile = X_raw.groupby('Cluster')[profile_cols].mean().round(3)
print("── Cluster Numeric Profile ──")
cluster_profile


By normalizing the distributions of Gender and First-Gen status, we can see if certain demographic groups are disproportionately represented in specific success clusters.

In [ ]:
# Categorical breakdown
print("── First-Generation Status by Cluster ──")
pd.crosstab(X_raw['Cluster'], X_raw['FIRST_GEN_STATUS'], normalize='index').round(3)


In [ ]:
print("\n── Gender by Cluster ──")
pd.crosstab(X_raw['Cluster'], X_raw['GENDER'], normalize='index').round(3)

Finally, we pull raw survey text from each cluster to hear the 'Student Voice' behind the numbers, allowing us to validate the mathematical groupings with qualitative context.

In [ ]:
# Sample comments from each cluster

ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')

print("── Representative Comments by Cluster ──")
for c in sorted(X_raw['Cluster'].unique()):
    index = X_raw[X_raw['Cluster'] == c].index
    subset = ML_Survey_Data.iloc[index,:]
    examples = subset.sample(min(2, len(subset)), random_state=42)['Free_Response_Text'].tolist()
    print(f"\nCluster {c} (n={len(subset)}):")
    for ex in examples:
        print(f"  • {ex}")

This gives us some insight into student perspective by cluster. We can combine this with topic modeling and sentiment analysis for even richer insight. Finally, let's look at aggregate numerical statistics by cluster:

In [ ]:
# Visualize numeric profile as a heatmap
import plotly.figure_factory as ff

z = cluster_profile.values.tolist()
x = cluster_profile.columns.tolist()
y = [f'Cluster {i}' for i in cluster_profile.index]

fig = px.imshow(
    cluster_profile,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='Blues',
    title='Cluster Profiles — Average Feature Values',
    labels={'x': 'Feature', 'y': 'Cluster', 'color': 'Mean Value'}
)
fig.show()


Pickle out the Cluster Model

In [ ]:
import pickle
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'

# Define the filename for the pickled model
kmeans_cluster_model = f'{model_filepath}kmeans_model_assignment.pkl'

# Pickle out the kmeans model
with open(kmeans_cluster_model, 'wb') as file:
    pickle.dump(kmeans, file)

print(f"KMeans model successfully pickled to {kmeans_cluster_model}")

## 10. Wrap-Up



**What you did:**
- Applied the **Elbow Method** to choose a reasonable K in the **K-Means** algorithm.
- Fit **K-Means** and assigned each student a cluster label
- Profiled clusters by GPA, DFW rates, demographic composition, and survey tone

**How to use clusters in IR practice:**
- Flag clusters with low GPA + high DFW for early-alert advising outreach
- Identify clusters dominated by subgroups such as first-gen students who express struggle — target programming
- Track cluster membership over cohorts to evaluate intervention effectiveness

**Key caution:** Clusters are patterns, not ground truth. Always validate with domain expertise and be cautious about using cluster labels in high-stakes decisions without further analysis.

